<a href="https://colab.research.google.com/github/1ashish1105/Build-GPT-from-scratch-Andrej-Karpathy-/blob/main/Build_GPT_from_scratch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Building ChatGPT

In [8]:
# We always start with a dataset to train on. Let's download the tiny shakespeare dataset
!wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt

--2026-07-16 17:36:03--  https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1115394 (1.1M) [text/plain]
Saving to: ‘input.txt.1’

input.txt.1         100%[===================>]   1.06M  --.-KB/s    in 0.05s   

2026-07-16 17:36:04 (22.5 MB/s) - ‘input.txt.1’ saved [1115394/1115394]



In [9]:
# read it in to inspect it
with open('input.txt', 'r', encoding='utf-8') as f:
    text = f.read()

In [11]:
len(text)

1115394

In [12]:
print(text[:1000])

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us kill him, and we'll have corn at our own price.
Is't a verdict?

All:
No more talking on't; let it be done: away, away!

Second Citizen:
One word, good citizens.

First Citizen:
We are accounted poor citizens, the patricians good.
What authority surfeits on would relieve us: if they
would yield us but the superfluity, while it were
wholesome, we might guess they relieved us humanely;
but they think we are too dear: the leanness that
afflicts us, the object of our misery, is as an
inventory to particularise their abundance; our
sufferance is a gain to them Let us revenge this with
our pikes, ere we become rakes: for the gods know I
speak this in hunger for bread, not in thirst for revenge.



In [17]:
chars=set(text)
print(char)
len(char)

{'!', ' ', 'b', ';', 'N', ':', 'u', 'g', 'l', 'p', 'A', 'S', 'q', 'J', '3', 'O', 'C', '-', '.', 'n', 'a', 'j', 'c', 'v', 'L', 'F', '?', 'W', 'Z', 'e', '$', 'm', 'I', 'f', 'D', 'x', 'U', 'R', 'k', 'Y', 'M', 'd', 'V', 'P', 'i', 'G', 'Q', 'H', 'o', 's', 'r', 't', "'", 'B', 'K', 'X', 'T', 'w', 'y', '&', 'h', 'E', 'z', ',', '\n'}


65

In [18]:
# create a mapping from characters to integers
stoi = { ch:i for i,ch in enumerate(chars) }
itos = { i:ch for i,ch in enumerate(chars) }
encode = lambda s: [stoi[c] for c in s] # encoder: take a string, output a list of integers
decode = lambda l: ''.join([itos[i] for i in l]) # decoder: take a list of integers, output a string

print(encode("hii there"))
print(decode(encode("hii there")))

[60, 44, 44, 1, 51, 60, 29, 50, 29]
hii there


In [21]:
import torch
data=torch.tensor(encode(text),dtype=torch.long)
print(data.shape)
print(data.dtype)
print(data[:1000])

torch.Size([1115394])
torch.int64
tensor([25, 44, 50, 49, 51,  1, 16, 44, 51, 44, 62, 29, 19,  5, 64, 53, 29, 33,
        48, 50, 29,  1, 57, 29,  1,  9, 50, 48, 22, 29, 29, 41,  1, 20, 19, 58,
         1, 33,  6, 50, 51, 60, 29, 50, 63,  1, 60, 29, 20, 50,  1, 31, 29,  1,
        49,  9, 29, 20, 38, 18, 64, 64, 10,  8,  8,  5, 64, 11,  9, 29, 20, 38,
        63,  1, 49,  9, 29, 20, 38, 18, 64, 64, 25, 44, 50, 49, 51,  1, 16, 44,
        51, 44, 62, 29, 19,  5, 64, 39, 48,  6,  1, 20, 50, 29,  1, 20,  8,  8,
         1, 50, 29, 49, 48,  8, 23, 29, 41,  1, 50, 20, 51, 60, 29, 50,  1, 51,
        48,  1, 41, 44, 29,  1, 51, 60, 20, 19,  1, 51, 48,  1, 33, 20, 31, 44,
        49, 60, 26, 64, 64, 10,  8,  8,  5, 64, 37, 29, 49, 48,  8, 23, 29, 41,
        18,  1, 50, 29, 49, 48,  8, 23, 29, 41, 18, 64, 64, 25, 44, 50, 49, 51,
         1, 16, 44, 51, 44, 62, 29, 19,  5, 64, 25, 44, 50, 49, 51, 63,  1, 58,
        48,  6,  1, 38, 19, 48, 57,  1, 16, 20, 44,  6, 49,  1, 40, 20, 50, 22,
      

In [23]:
n=int(0.9*len(data))
train_data=data[:n]
val_data=data[n:]

In [24]:
block_size=8
print(data[0:block_size+1])

tensor([25, 44, 50, 49, 51,  1, 16, 44, 51])
